<a href="https://colab.research.google.com/github/Saurabh312Kumar/Deep_learning/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt
from google.colab import drive

In [37]:
torch.manual_seed(42)

In [38]:
device=('cuda' if torch.cuda.is_available else 'cpu')
print(f"Device is {device}")

Device is cuda


In [39]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
train_df=pd.read_csv("/content/drive/MyDrive/mnist_train.csv")
test_df=pd.read_csv("/content/drive/MyDrive/mnist_test.csv")

In [41]:
print(train_df.shape)
print(test_df.shape)

(60000, 785)
(10000, 785)


In [42]:
train_df.head()

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [43]:
X_train=train_df.drop(columns=['label'],axis=1)
y_train=train_df['label']

X_test=test_df.drop(columns=['label'],axis=1)
y_test=test_df['label']

In [44]:
X_train=X_train/255.0
X_test=X_test/255.0

In [45]:
class MNISTDataset(Dataset):
  def __init__(self,feature,label):
    self.feature=torch.tensor(feature.values,dtype=torch.float32).reshape(-1,1,28,28)
    self.label=torch.tensor(label.values,dtype=torch.long)

  def __len__(self):
    return len(self.feature)

  def __getitem__(self,index):
    return self.feature[index],self.label[index]

In [46]:
train_dataset=MNISTDataset(X_train,y_train)
test_dataset=MNISTDataset(X_test,y_test)

In [47]:
train_loader=DataLoader(train_dataset,shuffle=True,batch_size=64,drop_last=True,pin_memory=True)
test_loader=DataLoader(test_dataset,shuffle=False,pin_memory=True)

In [48]:
class MyCnn(nn.Module):

  def __init__(self,input_feature):
    super().__init__()
    self.features=nn.Sequential(
        nn.Conv2d(input_feature,32,kernel_size=3,stride=2,padding=1),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(kernel_size=2,stride=2),

        nn.Conv2d(32,64,kernel_size=3,stride=2,padding=1),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )

    self.classifier=nn.Sequential(
        nn.Flatten(),
        nn.Linear(256,128),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(64,10)
    )

  def forward(self,x):
    x=self.features(x)
    x=self.classifier(x)

    return x

In [49]:
learning_rate=0.001
epochs=50

In [50]:
model=MyCnn(1)
model.to(device)
criterion=nn.CrossEntropyLoss()
optimizer=optim.RMSprop(model.parameters(),lr=learning_rate, weight_decay=1e-4)

In [52]:
for epoch in range(epochs):
  total_loss=0

  for batch_feature, batch_label in train_loader:

    batch_feature=batch_feature.to(device)
    batch_label=batch_label.to(device)

    outputs=model(batch_feature)

    loss=criterion(outputs,batch_label)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss+=loss.item()

  print(f"Epoch: {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader)}")

Epoch: 1/50, Loss: 0.24133193491199706
Epoch: 2/50, Loss: 0.09983594427749976
Epoch: 3/50, Loss: 0.07629060464475816
Epoch: 4/50, Loss: 0.06357314902916351
Epoch: 5/50, Loss: 0.05314776920553212
Epoch: 6/50, Loss: 0.04821669731184536
Epoch: 7/50, Loss: 0.048719642806180484
Epoch: 8/50, Loss: 0.041398317487378634
Epoch: 9/50, Loss: 0.03845515094653121
Epoch: 10/50, Loss: 0.03764809665585089
Epoch: 11/50, Loss: 0.03682731040553704
Epoch: 12/50, Loss: 0.033047411861679005
Epoch: 13/50, Loss: 0.03319877687664111
Epoch: 14/50, Loss: 0.030251828940644303
Epoch: 15/50, Loss: 0.027972729710024243
Epoch: 16/50, Loss: 0.027215512645819867
Epoch: 17/50, Loss: 0.0280803877545111
Epoch: 18/50, Loss: 0.02678362579573121
Epoch: 19/50, Loss: 0.026241086734776212
Epoch: 20/50, Loss: 0.02620994062527549
Epoch: 21/50, Loss: 0.025694025922414376
Epoch: 22/50, Loss: 0.023583363754854526
Epoch: 23/50, Loss: 0.026295637864698084
Epoch: 24/50, Loss: 0.023791471673107047
Epoch: 25/50, Loss: 0.02431870953470639

In [53]:
model.eval()

MyCnn(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=256, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [54]:
#testing
total=0
correct=0
with torch.no_grad():
  for batch_features,batch_label in test_loader:
    batch_features=batch_features.to(device)
    batch_label=batch_label.to(device)

    output=model(batch_features)
    _,predicted=torch.max(output,1)

    total=total+batch_label.shape[0]

    correct=correct + (predicted==batch_label).sum().item()

  print(f"Accuracy: {100*correct/total}%")


Accuracy: 98.89%


In [57]:
model.eval()
#training
total=0
correct=0

with torch.no_grad():

  for batch_feature,batch_label in train_loader:
    batch_feature=batch_feature.to(device)
    batch_label=batch_label.to(device)

    output=model(batch_feature)

    _,predicted=torch.max(output,1)

    total=total+batch_label.shape[0]
    correct=correct+(predicted==batch_label).sum().item()

  print(f"Accuracy: {100*correct/total}%")

Accuracy: 99.90828441835646%
